# Module 1: Getting Started
*From Molecular Dynamics to Machine Learning: An Introductory HPC Tutorial for Materials Science*

## Overview
The purpose of this tutorial is to act as a bridge between high-performance computing (HPC) resources and early-stage researchers interested in understanding how these tools can advance their work. It follows one materials science problem, the graphitization of amorphous carbon, studied using molecular dynamics simulations.

In Module 1, you will learn what an HPC system is and how to work on one, then run a small molecular dynamics simulation. This tutorial uses SDSC Expanse as its example system, but the same steps apply to most HPC clusters with minor changes. 

## Where this module fits
- **Module 1 (current):** run a molecular dynamics simulation on an HPC system
- **Module 2:** turn simulation output into an AI-ready dataset
- **Module 3:** train a baseline machine learning model on that dataset

## Learning objectives
By the end of this module, you will be able to:
- Log in to Expanse and check your allocation
- Navigate its file systems (home, project, scratch)
- Load software with modules
- Understand the input files for a ReaxFF simulation
- Submit and monitor a job with Slurm
- Run a LAMMPS simulation and locate its output
- Check that a simulation ran correctly by plotting its potential energy
## Before you start
**Prerequisites:** Some comfort with the command line is helpful but not required; basic commands are introduced as they come up

**You will need:** Access to an HPC system that uses the Slurm scheduler. Expanse allocations are available through ACCESS or the NAIRR Pilot. If you do not have HPC access, you can still follow along using the sample outputs in this repository.


## 1. What is an HPC system?

### Why HPC?
Many modern research problems, such as simulating atomic motion, predicting weather, and accelerating drug discovery, are too large, complex, or time-consuming for a standard computer. Molecular dynamics (MD) simulations are a prime example. Modeling the forces and motion of thousands to millions of atoms over time quickly exceeds what a laptop can do in a reasonable amount of time. High-performance computing (HPC) systems make problems manageable by dividing the work across many computers at once

### A brief history
Supercomputers emerged in the 1960s, driven by the Cold War, the nuclear arms race, and the space race. Early supercomputers were single, custom-built machines, but they became increasingly expensive while personal computers became cheaper. Scientists realized it was more cost-effective to connect many off-the-shelf computers than to build one multi-million-dollar machine, and this idea became the foundation of modern HPC.

### Anatomy of a cluster
A single computer in this network is called a **node**. Each node has its own processors, memory (RAM), and operating system, and nodes are connected through a high-speed network. A collection of connected nodes is called a **cluster**.

The main parts of a cluster you will interact with are:
- **Login nodes:** where you land upon logging in. They are shared by all users and are meant for editing files, organizing data, and submitting jobs. Do not run simulations here.
- **Compute nodes:** where the actual work happens. Each node contains many **cores** (individual processors), and jobs request a number of nodes and cores.
- **CPU and GPU nodes:** some compute nodes include GPUs (graphics processing units), which can greatly speed up certain calculations, including the ReaxFF simulations in this tutorial.
- **Shared storage:** all nodes can access the same file systems, so files you create on the login node are available to your jobs on the compute nodes.
- **The scheduler:** instead of running programs directly, you submit a request for resources, and a scheduler (on most clusters, **Slurm**) places your job in a queue and runs it when resources are available.

For a single program to use many cores or nodes at once, it must be written to split up the work. Most scientific codes like LAMMPS do this with **MPI** (Message Passing Interface). In an MD simulation, LAMMPS divides the simulation box into regions, assigns each region to a different processor, and has the processors exchange information about atoms near the region boundaries

Next, you will connect to a login node.

## 2. Connecting to and Navigating an HPC System

### The command line
Unlike your laptop, which uses a graphical user interface (windows, icons, and mouse clicks), HPC clusters are operated through a **command-line interface**. You interact with the cluster entirely through text commands typed into a terminal.
- **macOS/Linux:** open the Terminal application
- **Windows:** open PowerShell or Command Prompt

### Connecting with SSH
To connect to a remote cluster, you use **SSH (Secure Shell)**, a protocol that lets you send commands to a distant machine securely, as if you were sitting in front of it. SSH encrypts all transmitted data, protecting it from interception

To connect to Expanse: 
```bash
ssh username@login.expanse.sdsc.edu
```

You will be prompted for a password, and depending on the cluster, additional authentication may be needed. A successful connection places you on a login node, in your home directory with a prompt:

```

[username@login01 ~]$
```
If you are unsure where you are, type:
` pwd `
(print working directory), click enter, and you should expect the following output: `/home/your_username`


### Login Nodes vs Compute Nodes
- When you initially log on via SSH and land in `/home/your_username` you are on what is called a *login node*, which is a shared gateway that all users connect through
    - This is for file management and job submission only, not for running simulations or heavy workflows
    - Heavy computation happens on compute nodes, which are accessed through the job scheduler (covered later in this module)
- To navigate around the login node, it helps to know a few basic terminal commands:
| Command | What it does |
| :--- | :---|
| `pwd` | prints your current directory location | 
| `ls` | lists files and folders in the current directory | 
| `cd {folder}` | moves into a folder | 
| `cd` | returns to your home directory | 
| `cd ../` | go up one folder level | 
| `mkdir {folder}` | create a new directory | 
| `cp {file} {destination}` | copy a file | 
| `mv {file} {destination/new file name}` | move or rename a file | 
| `less {file}` | scrolls through a file (press `q` to quit) |
| `rm {file}` | delete a file | 
| `rm -r {folder}` | delete a folder and everything inside it | 

> **Warning:** There is no trash folder on a cluster. Anything deleted with `rm` is gone permanently.

### How the Filesystem is Organized

HPC clusters typically have multiple storage locations, each serving a different purpose. On Expanse, the three most important are as follows:
| Location | Path | Purpose | Notes |
|:-- | -- | -- | :--|
| Home Directory | `/home/your_username` | Scripts, small files, notebooks | Limited storage quota (100GB) - do not run jobs or store large datasets here |
| Project Storage | `/expanse/lustre/projects/your_project/your_username` | Simualtion data, large output files | Where actual research files should live |
| Scratch space | `/scratch/your_username` |Temporary files during active jobs | Scratch files will be purged 90 days from creation date |

* A common mistake is storing large simulation output files in your home directory and running out of quota - always write output to project storage
* Paths may vary by cluster, but most HPC systems follow the same pattern: a small, permanent home directory; larger project storage for active work; and scratch space for temporary files.

*Note: Always have an external backup of your data, as clusters generally do not automatically back up user data.*

### Checking your allocation
An allocation is the amount of compute time your project has been granted. Every job you run uses part of it, so it's good practice to check your balance before submitting jobs. On Expanse:
```bash
expanse-client user
```
This lists the projects you belong to and how much of each allocation remains.

> **On other systems:** Allocation commands differ between clusters. Check your cluster's user guide for the equivalent.

### Transferring Files

You will often need to move files between the cluster and your local machine. The most common tools are discussed below.

**`scp`**(secure copy) - best for simple file transfers

* Copying TO the cluster:

  `scp local_file.txt  username@login.expanse.sdsc.edu:/expanse/lustre/projects/your_project/`
    
* Copying FROM the cluster:

  `scp username@login.expanse.sdsc.edu:/expanse/lustre/projects/your_project/results.csv .`

  *Note: The `.` at the end means "copy here, into my current directory"*

**`rsync`** - better for large files or entire directories, can resume interrupted transfers
* Copying TO the cluster:

  `rsync -avz --partial --progress local_folder/ username@login.expanse.sdsc.edu:/expanse/lustre/projects/your_project/`
    
* Copying FROM the cluster:

  `rsync -{flags} --partial --progress username@login.expanse.sdsc.edu:/expanse/lustre/projects/your_project/results/ ./local_folder/`

* `rsync` transfers can be used with a variety of flags:

    | Flag | What it Does |
  | :-- | :-- |
    | -a | archive mode - preserves file structure, permissions, timestamps |
    | -v | verbose - shows what is being transferred on screen |
    | -z | compresses data during transfer, faster over slow connections |
    | --partial | keeps partially transferred files so you can resume if interrupted |
    | --progress | verbose - shows transfer progress and speed for each file |
    | --exclude='pattern' | skip files matching a pattern, e.g. `--exclude='*/restarts/'` to skip restart directories |
    | --dry-run | shows what would be transferred without actually doing it |

**Globus**
- `scp` and `rsync` work well for small to medium file transfers, but for very large datasets (hundreds of GB to TB), they can be unreliable - connections drop, transfers stall, and you are left manually restarting where you left off
- Globus is a research data transfer service specifically designed for large-scale transfers between HPC systems and personal storage
  - Unlike `scp` and `rsync`, Globus runs transfers in the background independently of your terminal session, so if your connection drops, the transfer automatically resumes without any intervention
  - SDSC Expanse has a dedicated Globus endpoint already set up, so no configuration is needed on the cluster side
- To use Globus:
  1. Create a free account at globus.org
  2. Install **Globus Connect Personal** on your local machine to make your laptop or external drive a Globus endpoint
  3. Log in and navigate to the **File Manager** tab
  4. Search for `SDSC HPC - Expanse Lustre` as one endpoint, and select your personal endpoint as the other
  5. Navigate to the files you want to transfer and click **start** - Globus will handle the rest
- Globus is the recommended tool for transferring large simulation output files, and it is worth setting up even if you only use it occasionally 

*Now that you know how to connect to the cluster, navigate its filesystem, and transfer files, the next step is actually setting up and running a simulation*

## 3. Getting the Software: Building LAMMPS

This tutorial uses LAMMPS, an open-source molecular dynamics code. LAMMPS is organized into optional **packages**, and the preinstalled version on many clusters, including Expanse, do not include the ReaxFF package used here, so you will likely need to build your own. This is the recommended route, as it is a great opportunity to practice building software and implement your troubleshooting skills.

The official LAMMPS documentation has thorough build instructions: [Building LAMMPS](https://docs.lammps.org/Build.html)

For this tutorial, your build needs:
- the **REAXFF** package: `-D PKG_REAXFF=on`
- x
- x

On Expanse, laod these modules before building:
- module purge
- module load
- x
- x

Once built, confirm ReaxFF is included:


> **On other systems:** Use `module spider` to find your cluster's compiler, MPI, and other necessary modules.
